In [3]:
# ---- Config (edit these three) ----
dataset = 'set_cover'
cfg_idx = 41111
eval_split = 'test'  # 'train', 'val', 'test', or 'all'
# ----------------------------------

import os
from pathlib import Path
import argparse
from typing import Any, Dict

import torch
import torch.nn.functional as F
import yaml
import tqdm

from utils import (
    load_data,
    load_model,
    load_json,
    save_json,
    set_seed,
    load_checkpoint,
    print_dash_str,
)


def _infer_feature_dimensions(train_loader):
    dataset = getattr(train_loader, 'dataset', None)
    if dataset is None or len(dataset) == 0:
        raise ValueError('Training dataset is empty; cannot infer feature dimensions.')
    sample = dataset[0]
    cons_nfeats = sample.constraint_features.shape[-1]
    edge_nfeats = sample.edge_attr.shape[-1]
    var_nfeats = sample.variable_features.shape[-1]
    return cons_nfeats, edge_nfeats, var_nfeats


def pad_tensor(input_, pad_sizes, pad_value=-1e8):
    max_pad_size = pad_sizes.max()
    output = input_.split(pad_sizes.cpu().numpy().tolist())
    output = torch.stack(
        [
            F.pad(slice_, (0, max_pad_size - slice_.size(0)), 'constant', pad_value)
            for slice_ in output
        ],
        dim=0,
    )
    return output


def evaluate_with_files(policy, data_loader, device, stats_filename, file_paths, accurate_path, inaccurate_path):
    mean_loss = 0
    mean_acc = 0
    mean_top5_acc = 0
    mean_score_diff = 0
    mean_normalized_score_diff = 0

    policy.eval()

    accurate_files = []
    inaccurate_files = []

    n_samples_processed = 0
    sample_offset = 0
    with torch.no_grad():
        for batch in tqdm.tqdm(data_loader, disable=True):
            batch = batch.to(device)
            logits = policy(
                batch.constraint_features,
                batch.edge_index,
                batch.edge_attr,
                batch.variable_features,
                candidates=batch.candidates,
                n_constraints_per_graph=batch.n_constraints_per_graph,
                n_variables_per_graph=batch.n_variables_per_graph,
            )
            logits = pad_tensor(logits[batch.candidates], batch.nb_candidates)
            loss = F.cross_entropy(logits, batch.candidate_choices)

            true_scores = pad_tensor(batch.candidate_scores, batch.nb_candidates).clip(0)
            true_bestscore = true_scores.max(dim=-1, keepdims=True).values

            predicted_bestindex = logits.max(dim=-1, keepdims=True).indices
            is_accurate = (
                true_scores.gather(-1, predicted_bestindex) == true_bestscore
            ).squeeze(-1)

            accuracy = is_accurate.float().mean().item()
            top5_acc = (
                true_scores.gather(-1, logits.topk(min(5, logits.size(-1))).indices)
                == true_bestscore
            ).float().max(dim=-1).values.mean().item()

            score_diff = (true_bestscore - true_scores.gather(-1, predicted_bestindex)).abs().mean().item()
            normalized_score_diff = ((true_bestscore - true_scores.gather(-1, predicted_bestindex)) / (true_bestscore + 1e-5)).mean().item()

            batch_size = batch.num_graphs
            mean_loss += loss.item() * batch_size
            mean_acc += accuracy * batch_size
            mean_top5_acc += top5_acc * batch_size
            mean_score_diff += score_diff * batch_size
            mean_normalized_score_diff += normalized_score_diff * batch_size
            n_samples_processed += batch_size

            batch_files = file_paths[sample_offset: sample_offset + batch_size]
            sample_offset += batch_size
            for fname, ok in zip(batch_files, is_accurate.tolist()):
                if ok:
                    accurate_files.append(Path(fname).name)
                else:
                    inaccurate_files.append(Path(fname).name)

    mean_loss /= n_samples_processed
    mean_acc /= n_samples_processed
    mean_top5_acc /= n_samples_processed
    mean_score_diff /= n_samples_processed
    mean_normalized_score_diff /= n_samples_processed

    instance_dir_results = {
        'Loss': mean_loss,
        'Accuracy': mean_acc,
        'Top5_Accuracy': mean_top5_acc,
        'Score_diff': mean_score_diff,
        'Normalized_score_diff': mean_normalized_score_diff,
        'n_samples': n_samples_processed,
    }

    save_json(stats_filename, instance_dir_results)
    save_json(accurate_path, accurate_files)
    save_json(inaccurate_path, inaccurate_files)
    return mean_loss, mean_acc, mean_top5_acc, mean_score_diff, mean_normalized_score_diff


def _load_config(config_root: Path, dataset: str, cfg_idx: int) -> Dict[str, Any]:
    cfg_path = config_root / f'{dataset}_{cfg_idx}'
    if not cfg_path.exists():
        raise FileNotFoundError(f'Configuration file not found: {cfg_path}')
    with open(cfg_path, 'r') as fh:
        cfg = yaml.safe_load(fh) or {}
    return cfg


def _merge_args_with_config(init_args, cfg: Dict[str, Any]):
    args_dict = {**cfg, **vars(init_args)}
    args = argparse.Namespace(**args_dict)
    args.model_id = f'{args.dataset}_cfg{args.cfg_idx}'
    args.device = 'cpu'
    return args


init_args = argparse.Namespace(
    dataset=dataset,
    cfg_idx=cfg_idx,
    config_root='cfg',
    model_suffix='',
    parent_test_stats_dir='data/results_summary/',
    eval_split=eval_split,
    eval_batch_size=32,
    train_split='train',
    val_split='valid',
    test_split='test',
)

cfg = _load_config(Path(init_args.config_root), init_args.dataset, init_args.cfg_idx)
args = _merge_args_with_config(init_args, cfg)

for key, value in vars(args).items():
    print(f'{key}: {value}')

set_seed(args.seed)
data = load_data(args, for_training=False)
cons_nfeats, edge_nfeats, var_nfeats = _infer_feature_dimensions(data.get('train'))

policy = load_model(args, cons_nfeats, edge_nfeats, var_nfeats)
print(f'Number of parameters: {sum(p.numel() for p in policy.parameters())}')

base_model_dir = Path(getattr(args, 'model_dir', './models'))
if getattr(args, 'model', None):
    base_model_dir = base_model_dir / args.model
model_id = getattr(args, 'model_id', None)
if model_id:
    base_model_dir = base_model_dir / model_id
model_suffix = getattr(args, 'model_suffix', '')
if model_suffix:
    base_model_dir = Path(f'{base_model_dir}_{model_suffix}')
assert os.path.exists(base_model_dir), f'Model directory does not exist: {base_model_dir}'

load_checkpoint(policy, None, step='max', save_dir=str(base_model_dir), device=args.device)
policy.eval()

model_name = base_model_dir.name
stats_root = Path(args.parent_test_stats_dir)
stats_root.mkdir(parents=True, exist_ok=True)
model_stats_root = stats_root / model_name
model_stats_root.mkdir(parents=True, exist_ok=True)

available_splits = ('train', 'val', 'test')
if args.eval_split == 'all':
    splits_to_evaluate = [split for split in available_splits if data.get(split) is not None]
else:
    splits_to_evaluate = [args.eval_split]

results = {}
for split in splits_to_evaluate:
    data_loader = data.get(split)
    if data_loader is None:
        print_dash_str(f"No data loader available for split '{split}', skipping.")
        continue
    split_stats_dir = model_stats_root / split
    split_stats_dir.mkdir(parents=True, exist_ok=True)
    stats_filename = split_stats_dir / 'eval_acc.json'
    accurate_path = split_stats_dir / 'accurate_files.json'
    inaccurate_path = split_stats_dir / 'inaccurate_files.json'
    print_dash_str(f'[{split}] Save stats to: {split_stats_dir}')

    file_paths = data.get(f'{split}_files', [])
    metrics = evaluate_with_files(
        policy,
        data_loader,
        args.device,
        str(stats_filename),
        file_paths,
        str(accurate_path),
        str(inaccurate_path),
    )
    results[split] = {'metrics': metrics, 'stats_path': stats_filename}
    print(
        f"[{split}] loss {metrics[0]:.4f}, acc {metrics[1]:.4f}, top5 {metrics[2]:.4f}, "
        f"score_diff {metrics[3]:.4f}, normalized {metrics[4]:.4f}"
    )

if args.eval_split == 'all' and results:
    aggregate_dir = model_stats_root / 'all'
    aggregate_dir.mkdir(parents=True, exist_ok=True)
    aggregate_path = aggregate_dir / 'eval_acc.json'

    aggregate_totals = {
        'Loss': 0.0,
        'Accuracy': 0.0,
        'Top5_Accuracy': 0.0,
        'Score_diff': 0.0,
        'Normalized_score_diff': 0.0,
    }
    total_samples = 0
    for split_result in results.values():
        split_stats = load_json(str(split_result['stats_path']))
        n_samples = split_stats.get('n_samples', 0)
        if n_samples == 0:
            continue
        total_samples += n_samples
        for key in aggregate_totals.keys():
            aggregate_totals[key] += split_stats[key] * n_samples

    if total_samples > 0:
        for key in aggregate_totals.keys():
            aggregate_totals[key] /= total_samples

    aggregate_stats = {
        **aggregate_totals,
        'n_samples': total_samples,
    }
    save_json(str(aggregate_path), aggregate_stats)
    print_dash_str('Aggregated results across all splits')
    print(
        f"[all] loss {aggregate_stats['Loss']:.4f}, acc {aggregate_stats['Accuracy']:.4f}, "
        f"top5 {aggregate_stats['Top5_Accuracy']:.4f}, score_diff {aggregate_stats['Score_diff']:.4f}, "
        f"normalized {aggregate_stats['Normalized_score_diff']:.4f}, samples {total_samples}"
    )


max_samples_per_split: None
config_root: cfg
resume_model_dir: 
model_suffix: 
model_dir: ./models
log_dir: ./logs
model: raw
hidden_channels: 256
num_layers: 3
n_breakings: 8
num_heads: 1
isab_num_inds: 50
sym_break_layers: 2
mp_layers: 2
use_set_transformer: True
breaking_selector_model_path: None
edge_nfeats: 1
use_default_features: False
remove_bad_candidates: True
file_pattern: sample_*.pkl
seed: 42
epochs: 250
lr: 0.0001
batch_size: 8
weight_decay: 0.0005
loss_option: LambdaNDCGLoss1
tier1_ub: 0.0
relevance_type: true_score
resume: False
eval_every: 100000
save_every: 100000
print_every: 100000
dataset_path: legacy_code_generator/data/samples/setcover/500r_1000c_0.05d
subsamples: 10overfit
dataset: set_cover
cfg_idx: 41111
parent_test_stats_dir: data/results_summary/
eval_split: test
eval_batch_size: 32
train_split: train
val_split: valid
test_split: test
model_id: set_cover_cfg41111
device: cpu
load dataset from legacy_code_generator/data/samples/setcover/500r_1000c_0.05d
using 

RuntimeError: Error(s) in loading state_dict for GNNPolicy:
	Unexpected key(s) in state_dict: "edge_embedding.0.weight", "edge_embedding.0.bias". 

In [ ]:
accurate_files = load_json(str(accurate_path))
inaccurate_files = load_json(str(inaccurate_path))

In [ ]:
len(inaccurate_files)

In [ ]:
from pathlib import Path
import torch

from utils import GraphDataset


def _resolve_file_path(file_name):
    file_path = Path(file_name)
    if file_path.exists():
        return file_path
    if eval_split == 'all':
        split_files = []
        for split in ('train', 'val', 'test'):
            split_files.extend(data.get(f'{split}_files', []))
    else:
        split_files = data.get(f'{eval_split}_files', [])
    for path in split_files:
        if Path(path).name == file_name:
            return Path(path)
    raise FileNotFoundError(f'No sample file found for {file_name}')


def get_input(file_name):
    # given a file_name, return a graph data. similar to get() in GraphDataset in utils.py
    file_path = _resolve_file_path(file_name)
    dataset = GraphDataset(
        sample_files=[file_path],
        edge_nfeats=getattr(args, 'edge_nfeats', 1),
        args=args,
    )
    return dataset.get(0)


def get_prediction(inputs):
    inputs = inputs.to(args.device)
    n_constraints = torch.tensor([inputs.n_constraints_per_graph], device=args.device)
    n_variables = torch.tensor([inputs.n_variables_per_graph], device=args.device)
    logits = policy(
        inputs.constraint_features,
        inputs.edge_index,
        inputs.edge_attr,
        inputs.variable_features,
        candidates=inputs.candidates,
        n_constraints_per_graph=n_constraints,
        n_variables_per_graph=n_variables,
    )
    return logits


def get_predicted_scores(logits, inputs):
    logits = logits[inputs.candidates]
    return logits


def get_true_scores(inputs):
    true_scores = inputs.candidate_scores
    return true_scores


def get_score_margins(scores):
    max_scores = scores.max(dim=-1, keepdim=True).values
    return max_scores - scores


def get_norm_scores(scores):
    max_scores = scores.max(dim=-1, keepdim=True).values
    return torch.where(max_scores > 0, scores / max_scores, scores)


def get_ranks(logits, inputs):
    scores = inputs.candidate_scores
    predicted_scores = logits
    candidates = inputs.candidates
    ranks = {}
    for idx in range(scores.numel()):
        score = scores[idx]
        predicted_score = predicted_scores[idx]
        rank = int((scores > score).sum().item()) + 1
        predicted_rank = int((predicted_scores > predicted_score).sum().item()) + 1
        ranks[idx] = {
            'candidate_id': int(candidates[idx].item()),
            'rank': rank,
            'predicted_rank': predicted_rank,
        }
    return ranks


In [ ]:
def _second_best_margin(scores):
    top2 = torch.topk(scores, k=2).values
    return top2[0] - top2[1]

def _avg_margins(file_list):
    margins = []
    with torch.no_grad():
        for file_name in file_list:
            inputs = get_input(file_name)
            true_scores = get_true_scores(inputs)
            true_norm_scores = get_norm_scores(true_scores)
            margins.append(_second_best_margin(true_norm_scores).item())
    if not margins:
        return float('nan')
    return sum(margins) / len(margins)

accurate_avg_margin = _avg_margins(accurate_files)
inaccurate_avg_margin = _avg_margins(inaccurate_files)
print(f'avg true_norm_margin (accurate): {accurate_avg_margin:.6f}')
print(f'avg true_norm_margin (inaccurate): {inaccurate_avg_margin:.6f}')

mismatch_rows = []
with torch.no_grad():
    for file_name in inaccurate_files:
        inputs = get_input(file_name)
        logits = get_prediction(inputs)
        predicted_scores = get_predicted_scores(logits, inputs)
        ranks = get_ranks(predicted_scores, inputs)
        for idx, info in ranks.items():
            rank = info['rank']
            predicted_rank = info['predicted_rank']
            if (rank == 1) != (predicted_rank == 1):
                mismatch_rows.append({
                    'file_name': file_name,
                    'idx': idx,
                    'candidate_id': info['candidate_id'],
                    'rank': rank,
                    'predicted_rank': predicted_rank,
                })

print(f'mismatches (rank==1 xor predicted_rank==1): {len(mismatch_rows)}')
mismatch_rows[:10]

In [ ]:
best_second_pairs = []
for i in range(0, len(mismatch_rows), 2):
    pair = mismatch_rows[i:i+2]
    if len(pair) < 2:
        continue
    if pair[0]['file_name'] != pair[1]['file_name']:
        continue
    if pair[0]['rank'] == 1:
        best_row, second_row = pair[0], pair[1]
    elif pair[1]['rank'] == 1:
        best_row, second_row = pair[1], pair[0]
    else:
        continue
    best_second_pairs.append({
        'file_name': best_row['file_name'],
        'best_candidate_idx': best_row['idx'],
        'second_best_candidate_idx': second_row['idx'],
    })

best_second_pairs[:10]

In [ ]:
for pair in best_second_pairs:
    sample = pair["file_name"]
    best_candidate_idx = pair["best_candidate_idx"]
    second_best_candidate_idx = pair["second_best_candidate_idx"]
    inputs = get_input(sample)
    predicted_scores = get_predicted_scores(get_prediction(inputs), inputs)
    predicted_norm_scores = get_norm_scores(predicted_scores)
    true_scores = get_true_scores(inputs)
    true_norm_scores = get_norm_scores(true_scores)
    true_margins = get_score_margins(true_scores)
    true_norm_margins = get_score_margins(true_norm_scores)
    ranks = get_ranks(predicted_scores, inputs)
    # print("%.2f" % predicted_scores[best_candidate_idx].item(), "%.2f" % predicted_scores[second_best_candidate_idx].item())
    # print("%.2f" % true_scores[best_candidate_idx].item(), "%.2f" % true_scores[second_best_candidate_idx].item())
    best_node_input = inputs.variable_features[inputs.candidates][best_candidate_idx]
    second_best_node_input = inputs.variable_features[inputs.candidates][second_best_candidate_idx]
    diff = best_node_input - second_best_node_input
    print(f"different features happen at the {torch.where(diff!=0)[0].tolist()}th features")
    print([round(d,4) for d in diff[diff!=0].tolist()])

0: "type_0", 

1: "type_1", 

2: "type_2", 

3: "type_3", 

4: "has_lb",

5: "has_ub",
 
6: "sol_is_at_lb", 

7: "sol_is_at_ub", 

8: "sol_frac",

9: "coef_normalized", 

10: "sol_val"

In [ ]:
batch.n_variables_per_graph